In [8]:
import logging 
import os
import sys

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
try:
    project_root_directory = Path(__file__).resolve().parent.parent
except NameError:
    project_root_directory = Path.cwd().parent

def setup_logger(file_path: str | None = None, level=logging.INFO) -> logging.Logger:
    try:
        if file_path is None:
            file_path = os.getenv('LOG_FILE_PATH')

        if not file_path:
            raise ValueError('No file path has provided!')

        log_path = Path(file_path)

        if not log_path.is_absolute():
            log_path = project_root_directory / log_path

        log_path.parent.mkdir(parents=True, exist_ok=True)

        root_logger = logging.getLogger()
        if root_logger.handlers:
            return logging.getLogger(__name__)

        logging.basicConfig(
            level=level,
            format='%(asctime)s | %(funcName)s | %(levelname)s | %(message)s',
            handlers=[
                logging.FileHandler(log_path, encoding='utf-8'),
                logging.StreamHandler(sys.stdout)
            ],
            force=True
        )

        return logging.getLogger(__name__)
        
    except Exception as error:
        print(f"Failed to setup logs due to {error}")

logger = setup_logger()

In [9]:
def main():
    logger.info("Starting pipeline")

main()

2026-09-27 11:55:24,081 | main | INFO | Starting pipeline


In [10]:
import sys

print(sys.executable)

c:\dvd-rental-modern-data-platform\.venv\Scripts\python.exe


In [16]:
# Developing the settings.py

import os

from dotenv import load_dotenv

load_dotenv()

AWS_REGION = os.getenv('AWS_REGION')
S3_BRONZE_BUCKET = os.getenv('S3_BRONZE_BUCKET')
RAW_DATA_PATH = os.getenv('RAW_DATA_PATH')

def validate_settings():

    try:
        required_settings = {
            'AWS_REGION': AWS_REGION,
            'S3_BRONZE_BUCKET': S3_BRONZE_BUCKET,
            'RAW_DATA_PATH': RAW_DATA_PATH
        }

        missing_settings = [name for name, value in required_settings.items() if not value]

        logger.info(f"Missing settings: {missing_settings}")

        if missing_settings:
            raise ValueError(f"Missing required settings: {', '.join(missing_settings)}")
        
    except Exception as error:
        logger.error(f"Settings validation failed due to {error}")
        raise

validate_settings()

2026-09-27 11:58:06,938 | validate_settings | INFO | Missing settings: []


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone

@dataclass(frozen=True)
class BatchContext:
    batch_id: str
    ingestion_timestamp: datetime
    ingestion_date: str

def batch_conext() -> BatchContext:

    ingestion_timestamp = datetime.now(timezone.utc)
    batch_id = ingestion_timestamp.strftime('%Y%m%d_%H%M%S')
    ingestion_date = ingestion_timestamp.strftime('%Y%m%d')

    return BatchContext(
        batch_id = batch_id,
        ingestion_timestamp = ingestion_timestamp,
        ingestion_date = ingestion_date
    )

In [6]:
try:
    print(hello)
except Exception as error:
    raise RuntimeError(f"Failed due to {error}")

RuntimeError: Failed due to name 'hello' is not defined

In [2]:
print(hello)

NameError: name 'hello' is not defined

In [7]:
from dataclasses import dataclass

try:
    @dataclass(frozen=True)
    class BatchContext:
        ingestion_timestamp: datetime
except Exception as error:
    raise RuntimeError(f"Failed due to {error}") from error

RuntimeError: Failed due to name 'datetime' is not defined

In [8]:
from dataclasses import dataclass

@dataclass(frozen=True)
class BatchContext:
    ingestion_timestamp: datetime

NameError: name 'datetime' is not defined

In [16]:
from pathlib import Path

def discover_files(data_path: str | None = None) -> list[Path]:

    project_root_dir = Path.cwd().parent

    raw_path = project_root_dir / Path(data_path)

    print(raw_path)

    if not raw_path.exists():
        raise FileNotFoundError(f"Raw data directory does not exist: {raw_path}")

    if not raw_path.is_dir():
        raise NotADirectoryError(f"Raw data path is not a directory: {raw_path}")

    csv_files = sorted(raw_path.glob("*.csv"))

    print(csv_files)

    if not csv_files:
        raise FileNotFoundError*(f"No CSV files found in: {raw_path}")

    return csv_files

print(discover_files())

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

In [18]:
from pathlib import Path

raw = Path.cwd().parent

print(raw)

c:\dvd-rental-modern-data-platform


In [11]:
import boto3
import sys

from pathlib import Path
from botocore.exceptions import ClientError

PROJECT_ROOT_DIR = Path.cwd().parent
if str(PROJECT_ROOT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR))

from src.utils.setup_logger import logger

class S3Client:
    def __init__(self, region_name: str):
        self.client = boto3.client('s3', region_name=region_name)

    def upload_file(self, local_path: Path, bucket: str, s3_key: str) -> None:

        try:
            file_path = Path(local_path)

            if not file_path.is_file():
                raise FileNotFoundError(f"Local file not found: {file_path}")

            logger.info(f"File uploading to s3: {str(file_path)}")

            self.client.upload_file(
                str(file_path),
                bucket,
                s3_key,
                ExtraArgs={
                    "ContentType": "text/csv"
                }
            )

            logger.info(f"File:{file_path.name} uploaded successfully to S3://{bucket}/{s3_key}")

        except ClientError as error:
            logger.error(f"Filed to upload file: {file_path.name} due to {error}")
            raise

s3_client = S3Client(region_name="us-east-1")

s3_client.upload_file(Path('C:/dvd-rental-modern-data-platform/data/raw/address.csv'), 'dvd-rental-modern-data-platform-bronze', 'dvd-rental-modern-data-platform-bronze/address')

2026-09-30 23:05:32,798 | load | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-09-30 23:05:33,143 | upload_file | INFO | File uploading to s3: C:\dvd-rental-modern-data-platform\data\raw\address.csv


S3UploadFailedError: Failed to upload C:\dvd-rental-modern-data-platform\data\raw\address.csv to dvd-rental-modern-data-platform-bronze/dvd-rental-modern-data-platform-bronze/address: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:iam::851382689558:user/ecommerce-sales-etl-and-analysis-user is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::dvd-rental-modern-data-platform-bronze/dvd-rental-modern-data-platform-bronze/address" because no identity-based policy allows the s3:PutObject action

In [15]:
def upload(text: str, num: int | None = 10) -> list[int]:
    print(num, text)

    return str(num) + str(text)

a = upload(text=1)
print(a)

10 1
101


In [21]:
from pathlib import Path

a = print(type(Path('C:/dvd-rental-modern-data-platform/data/raw/actor.csv')))

if type(a) == Path:
    print('Path')

<class 'pathlib._local.WindowsPath'>
